# **Load Silver Layer (Bronze -> Silver)**

**Purpose:** 
	This notebook performs ETL (Extract, Transform, Load) process to 
	insert data into the silver tables from bronze schema.
	This performs below tasks:
		
        - Truncate silver tables before loading data
        - Inserts cleansed, transformed data from bronze to silver tables


# **For all CRM tables**

In [0]:
%sql
--truncating and inserting data into datalakehouse.silver.crm_cust_info
TRUNCATE TABLE datalakehouse.silver.crm_cust_info;
INSERT INTO datalakehouse.silver.crm_cust_info (cst_id, cst_key, cst_firstname, cst_lastname, cst_marital_status, cst_gndr, cst_create_date)
SELECT cst_id, 
       cst_key, 
       TRIM(cst_firstname) AS cst_firstname, 
       TRIM(cst_lastname) AS cst_lastname, 
       CASE
        WHEN cst_marital_status = 'M'THEN 'Merried'
        WHEN cst_marital_status = 'S'THEN 'Single'
        ELSE 'n/a' 
       END AS cst_matrial_status, 
       CASE
        WHEN cst_gndr = 'M'THEN 'Male'
        WHEN cst_gndr = 'F'THEN 'Female'
        ELSE 'n/a' 
       END AS cst_gndr,
       cst_create_date
FROM (
    SELECT cst_id, cst_key, cst_firstname, cst_lastname, cst_marital_status, cst_gndr, cst_create_date,
        row_number() OVER(partition by cst_id order by cst_create_date DESC) as rn
    FROM datalakehouse.bronze.crm_cust_info
    WHERE cst_id IS NOT NULL)t
WHERE rn = 1;

--truncating and inserting data into datalakehouse.silver.crm_prd_info     
TRUNCATE TABLE datalakehouse.silver.crm_prd_info;
INSERT INTO datalakehouse.silver.crm_prd_info (prd_id, cat_id, prd_key, prd_name, prd_cost, prd_line, prd_start_dt, prd_end_dt)
SELECT 
    prd_id, 
    REPLACE(substring(prd_key, 1, 5), '-', '_') AS cat_id, 
    substring(prd_key, 7, len(prd_key)) AS prd_key, 
    prd_name, 
    IFNULL(prd_cost, 0) AS prd_cost, 
    CASE UPPER(TRIM(prd_line)) --This case when statement implemented readability for row cust_gender
		WHEN 'M' THEN 'Mountain'
		WHEN 'R' THEN 'Road'
		WHEN 'S' THEN 'Other Sales'
		WHEN 'T' THEN 'Touring'
		ELSE 'N/A'
	END prd_line, 
    CAST(prd_start_dt AS DATE) AS prd_start_dt, 
    CAST(lead(prd_start_dt) over(partition by prd_key order by prd_start_dt) - 1  AS DATE) AS prd_end_dt
FROM datalakehouse.bronze.crm_prd_info;
        
--truncating and inserting data into datalakehouse.silver.crm_sales_details
TRUNCATE TABLE datalakehouse.silver.crm_sales_details;
INSERT INTO datalakehouse.silver.crm_sales_details (sls_ord_num, sls_prd_key, sls_cust_id, sls_order_dt, sls_ship_dt, sls_due_dt, sls_sales, sls_quantity, sls_price)
SELECT sls_ord_num, sls_prd_key, sls_cust_id, 
    CASE
        WHEN sls_order_dt <= 0 or LEN(sls_order_dt) !=8 THEN NULL
        ELSE TO_DATE(CAST(sls_order_dt AS VARCHAR(50)), 'yyyyMMdd') 
    END AS sls_order_dt, 
    CASE
        WHEN sls_ship_dt <= 0 or LEN(sls_ship_dt) !=8 THEN NULL
        ELSE TO_DATE(CAST(sls_ship_dt AS VARCHAR(50)), 'yyyyMMdd') 
    END AS sls_ship_dt, 
    CASE
        WHEN sls_due_dt <= 0 or LEN(sls_due_dt) !=8 THEN NULL
        ELSE TO_DATE(CAST(sls_due_dt AS VARCHAR(50)), 'yyyyMMdd') 
    END AS sls_due_dt, 
    CASE
        WHEN sls_sales < 0 OR sls_sales IS NULL OR sls_sales != sls_price * sls_quantity
        THEN sls_price * sls_quantity
        ELSE sls_sales
    END AS sls_sales,
    sls_quantity, 
    CASE
        WHEN sls_price < 0 OR sls_price IS NULL
        THEN sls_sales / NULLIF(sls_quantity, 0)
        ELSE sls_price
    END AS sls_price
FROM datalakehouse.bronze.crm_sales_details

# **For all ERP tables**

In [0]:
%sql
--truncating and inserting data into datalakehouse.silver.erp_cust_az12
TRUNCATE TABLE datalakehouse.silver.erp_cust_az12;
INSERT INTO datalakehouse.silver.erp_cust_az12 (cid, bdate, gen)
SELECT CASE 
        WHEN cid like 'NAS%' THEN substring(cid, 4, len(cid))
        ELSE cid 
       END AS cid, 
    CASE 
       WHEN bdate < '1915-12-31' or bdate > current_date() THEN NULL
       ELSE bdate
    END AS bdate, 
    CASE 
        WHEN UPPER(TRIM(gen)) = 'M' OR UPPER(TRIM(gen)) = 'MALE' THEN 'Male'
        WHEN UPPER(TRIM(gen)) = 'F' OR UPPER(TRIM(gen)) = 'FEMALE'THEN 'Female'
        ELSE 'n/a'
    END AS gen   
FROM datalakehouse.bronze.erp_cust_az12;

--truncating and inserting data into datalakehouse.silver.erp_loc_a101
TRUNCATE TABLE datalakehouse.silver.erp_loc_a101;
INSERT INTO datalakehouse.silver.erp_loc_a101 (cid, cntry)
SELECT  
    REPLACE(cid,'-', '') AS cid, 
    CASE UPPER(TRIM(cntry))
        WHEN 'US' THEN 'United States'
        WHEN 'USA' THEN 'United States'
        WHEN 'DE' THEN 'Germany'
        WHEN '' THEN 'n/a'
        ELSE COALESCE(TRIM(cntry), 'n/a')
    END AS cntry
FROM datalakehouse.bronze.erp_loc_a101;

--truncating and inserting data into datalakehouse.silver.erp_px_cat_glv2  
TRUNCATE TABLE datalakehouse.silver.erp_px_cat_glv2;
INSERT INTO datalakehouse.silver.erp_px_cat_glv2 (id, cat, subcat, maintenance)
SELECT  id, cat, subcat, maintenance
FROM datalakehouse.bronze.erp_px_cat_glv2